In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Load the data
df = pd.read_csv('room_classification_features.csv')

# Combine blinds_up and blinds_down into "meeting_room"
df['room_label'] = df['room_label'].replace({
    'blinds_up': 'meeting_room',
    'blinds_down': 'meeting_room'
})

X = df.drop(columns=['room', 'room_label', 'trial'])
y = df['room_label']

# Update room column for stratification (combine blinds variants)
stratify_col = df['room'].replace({
    'blinds_up': 'meeting_room_base',
    'blinds_up_motion': 'meeting_room_motion',
    'blinds_up_person': 'meeting_room_person',
    'blinds_up_object': 'meeting_room_object',
    'blinds_down': 'meeting_room_base',
    'blinds_down_motion': 'meeting_room_motion',
    'blinds_down_person': 'meeting_room_person',
    'blinds_down_object': 'meeting_room_object',
})

In [19]:
def calculate_metrics(y_true, y_pred):
    metrics = {}

    # Overall accuracy
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    # Per-class accuracy
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    
    metrics['hallway_accuracy'] = report['hallway']['recall']
    metrics['kitchen_accuracy'] = report['kitchen']['recall']
    metrics['lab_accuracy'] = report['lab']['recall']
    metrics['meeting_room_accuracy'] = report['meeting_room']['recall']
    
    return metrics

In [6]:
# Define feature groups by prefix
uo_features = [col for col in X.columns if col.startswith('UO')]
graphics_features = [col for col in X.columns if col.startswith('Graphics')]
amo_features = [col for col in X.columns if col.startswith('AMO')]
cpu_features = [col for col in X.columns if col.startswith('CPU')]

# Verify counts
print(f"UO features: {len(uo_features)}")
print(f"Graphics features: {len(graphics_features)}")
print(f"AMO features: {len(amo_features)}")
print(f"CPU features: {len(cpu_features)}")
print(f"Total: {len(uo_features) + len(graphics_features) + len(amo_features) + len(cpu_features)}")
print(f"All features: {len(X.columns)}")

UO features: 116
Graphics features: 36
AMO features: 36
CPU features: 6
Total: 194
All features: 194


In [7]:
# PHASE 1, features are separated by their file of origin

# run the 4 combinations of the model with only 1/4 categories
# run all combinations of the model with only 2/4 categories
# run the 4 combinations of the model with only 3/4 categories

# categories
# features that start with UO (features from UnityObjects.csv)
# features that start with Graphics (features from Graphics.csv)
# features that start with AMO (features from AllManagedObjects.csv)
# features that start with CPU (Unity "power" profiler)

def ablation_study_by_source(X, y, stratify_col):
    results = {}
    
    # Baseline with all features, expecting 100% accuracy
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=0.25, 
        random_state=42, 
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Baseline (All Features)'] = calculate_metrics(y_test, y_pred)

    # 1/4 categories (4 models)
    X_uo = X[uo_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_uo, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only Unity Objects'] = calculate_metrics(y_test, y_pred)
    
    X_graphics = X[graphics_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_graphics, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only Graphics'] = calculate_metrics(y_test, y_pred)
    
    X_amo = X[amo_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_amo, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only Managed Objects'] = calculate_metrics(y_test, y_pred)
    
    X_cpu = X[cpu_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_cpu, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only CPU'] = calculate_metrics(y_test, y_pred)

    # 2/4 categories (6 combinations)
    X_uo_graphics = X[uo_features + graphics_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_uo_graphics, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['UO + Graphics'] = calculate_metrics(y_test, y_pred)
    
    X_uo_amo = X[uo_features + amo_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_uo_amo, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['UO + AMO'] = calculate_metrics(y_test, y_pred)
    
    X_uo_cpu = X[uo_features + cpu_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_uo_cpu, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['UO + CPU'] = calculate_metrics(y_test, y_pred)
    
    X_graphics_amo = X[graphics_features + amo_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_graphics_amo, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Graphics + AMO'] = calculate_metrics(y_test, y_pred)
    
    X_graphics_cpu = X[graphics_features + cpu_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_graphics_cpu, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Graphics + CPU'] = calculate_metrics(y_test, y_pred)
    
    X_amo_cpu = X[amo_features + cpu_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_amo_cpu, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['AMO + CPU'] = calculate_metrics(y_test, y_pred)

    # 3/4 categories (4 models)
    X_no_uo = X[graphics_features + amo_features + cpu_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_uo, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['All except UO'] = calculate_metrics(y_test, y_pred)
    
    X_no_graphics = X[uo_features + amo_features + cpu_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_graphics, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['All except Graphics'] = calculate_metrics(y_test, y_pred)
    
    X_no_amo = X[uo_features + graphics_features + cpu_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_amo, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['All except AMO'] = calculate_metrics(y_test, y_pred)
    
    X_no_cpu = X[uo_features + graphics_features + amo_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_cpu, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['All except CPU'] = calculate_metrics(y_test, y_pred)
    
    return results

In [8]:
phase1_results = ablation_study_by_source(X, y, stratify_col)
phase1_df = pd.DataFrame(phase1_results).T
phase1_df_sorted = phase1_df.sort_values('accuracy', ascending=False)
phase1_df_sorted

,accuracy,hallway_accuracy,kitchen_accuracy,lab_accuracy,meeting_room_accuracy
Baseline (All Features),1.000000,1.0,1.0,1.000000,1.000000
Only Unity Objects,1.000000,1.0,1.0,1.000000,1.000000
Only Graphics,1.000000,1.0,1.0,1.000000,1.000000
Only Managed Objects,1.000000,1.0,1.0,1.000000,1.000000
UO + Graphics,1.000000,1.0,1.0,1.000000,1.000000
UO + AMO,1.000000,1.0,1.0,1.000000,1.000000
UO + CPU,1.000000,1.0,1.0,1.000000,1.000000
All except AMO,1.000000,1.0,1.0,1.000000,1.000000
Graphics + AMO,1.000000,1.0,1.0,1.000000,1.000000
AMO + CPU,1.000000,1.0,1.0,1.000000,1.000000


In [10]:
def ablation_study_by_furniture(X, y, stratify_col):
    results = {}
    
    furniture_categories = ['BED', 'COUCH', 'LAMP', 'OTHER', 'PLANT', 'SCREEN', 'STORAGE', 'TABLE', 'WALL_ART']
    architecture_categories = ['CEILING', 'DOOR_FRAME', 'FLOOR', 'WALL_FACE', 'INVISIBLE_WALL_FACE', 'WINDOW_FRAME']
    
    furniture_cols = [col for col in X.columns if any(cat in col for cat in furniture_categories)]
    X_no_furniture = X.drop(columns=furniture_cols)
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_furniture, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['No Furniture'] = calculate_metrics(y_test, y_pred)
    
    architecture_cols = [col for col in X.columns if any(cat in col for cat in architecture_categories)]
    X_no_architecture = X.drop(columns=architecture_cols)
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_architecture, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['No Architecture'] = calculate_metrics(y_test, y_pred)
    
    all_object_cols = [col for col in X.columns if any(cat in col for cat in furniture_categories + architecture_categories)]
    X_no_objects = X.drop(columns=all_object_cols)
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_objects, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['No Furniture or Architecture'] = calculate_metrics(y_test, y_pred)

    return results

In [11]:
furniture_study_results = ablation_study_by_furniture(X, y, stratify_col)
furniture_removal_df = pd.DataFrame(furniture_study_results).T
furniture_sorted = furniture_removal_df.sort_values('accuracy', ascending=False)
furniture_sorted

,accuracy,hallway_accuracy,kitchen_accuracy,lab_accuracy,meeting_room_accuracy
No Furniture,1.0,1.0,1.0,1.0,1.0
No Architecture,1.0,1.0,1.0,1.0,1.0
No Furniture or Architecture,1.0,1.0,1.0,1.0,1.0


In [12]:
def ablation_study_phase2(X, y, stratify_col):
    results = {}
    
    uo_type_features = [col for col in X.columns if col.startswith('UO_Type_')]
    X_uo_type = X[uo_type_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_uo_type, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only UO_Type (Generic Unity Objects)'] = calculate_metrics(y_test, y_pred)
    
    uo_name_features = [col for col in X.columns if col.startswith('UO_Name_')]
    X_uo_name = X[uo_name_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_uo_name, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only UO_Name (Named Objects)'] = calculate_metrics(y_test, y_pred)
    
    count_features = [col for col in X.columns if '_count' in col]
    X_no_count = X.drop(columns=count_features)
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_count, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['No Count Features'] = calculate_metrics(y_test, y_pred)
    
    allocated_features = [col for col in X.columns if 'allocated' in col]
    X_no_allocated = X.drop(columns=allocated_features)
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_allocated, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['No Allocated Features'] = calculate_metrics(y_test, y_pred)
    
    global_mesh_features = [col for col in X.columns if 'GLOBAL_MESH' in col]
    X_no_global = X.drop(columns=global_mesh_features)
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_global, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['No GLOBAL_MESH'] = calculate_metrics(y_test, y_pred)
    
    return results

In [13]:
phase2_results = ablation_study_phase2(X, y, stratify_col)
phase2_df = pd.DataFrame(phase2_results).T
phase2_df_sorted = phase2_df.sort_values('accuracy', ascending=False)
phase2_df_sorted

,accuracy,hallway_accuracy,kitchen_accuracy,lab_accuracy,meeting_room_accuracy
Only UO_Type (Generic Unity Objects),1.0,1.0,1.0,1.0,1.0
Only UO_Name (Named Objects),1.0,1.0,1.0,1.0,1.0
No Count Features,1.0,1.0,1.0,1.0,1.0
No Allocated Features,1.0,1.0,1.0,1.0,1.0
No GLOBAL_MESH,1.0,1.0,1.0,1.0,1.0


In [14]:
def ablation_study_phase3_extreme(X, y, stratify_col):
    results = {}
    
    single_uo_type = ['UO_Type_Mesh_allocated_sum']
    X_single = X[single_uo_type]
    X_train, X_test, y_train, y_test = train_test_split(
        X_single, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Single Feature: UO_Type_Mesh_allocated_sum'] = calculate_metrics(y_test, y_pred)
    
    single_gameobject = ['UO_Type_GameObject_allocated_count']
    X_single = X[single_gameobject]
    X_train, X_test, y_train, y_test = train_test_split(
        X_single, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Single Feature: UO_Type_GameObject_allocated_count'] = calculate_metrics(y_test, y_pred)
    
    sum_features = [col for col in X.columns if col.endswith('_sum')]
    X_sums = X[sum_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_sums, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only Sum Features'] = calculate_metrics(y_test, y_pred)
    
    mean_features = [col for col in X.columns if col.endswith('_mean')]
    X_means = X[mean_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_means, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only Mean Features'] = calculate_metrics(y_test, y_pred)
    
    count_features_only = [col for col in X.columns if col.endswith('_count')]
    X_counts = X[count_features_only]
    X_train, X_test, y_train, y_test = train_test_split(
        X_counts, y, 
        test_size=0.25, 
        random_state=42,
        stratify=stratify_col
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only Count Features'] = calculate_metrics(y_test, y_pred)
    
    return results

In [15]:
phase3_results = ablation_study_phase3_extreme(X, y, stratify_col)
phase3_df = pd.DataFrame(phase3_results).T
phase3_df_sorted = phase3_df.sort_values('accuracy', ascending=False)
phase3_df_sorted

,accuracy,hallway_accuracy,kitchen_accuracy,lab_accuracy,meeting_room_accuracy
Single Feature: UO_Type_Mesh_allocated_sum,1.0,1.0,1.0,1.0,1.0
Single Feature: UO_Type_GameObject_allocated_count,1.0,1.0,1.0,1.0,1.0
Only Sum Features,1.0,1.0,1.0,1.0,1.0
Only Mean Features,1.0,1.0,1.0,1.0,1.0
Only Count Features,1.0,1.0,1.0,1.0,1.0


In [20]:
def ablation_study_phase4_random(X, y, stratify_col):
    results = {}
    
    np.random.seed(42)
    for i in range(10):
        random_5_features = list(np.random.choice(X.columns, size=5, replace=False))
        X_random = X[random_5_features]
        X_train, X_test, y_train, y_test = train_test_split(
            X_random, y, 
            test_size=0.25, 
            random_state=42,
            stratify=stratify_col
        )
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        rf_model.fit(X_train_scaled, y_train)
        y_pred = rf_model.predict(X_test_scaled)
        results[f'Random 5 Features (trial {i+1})'] = calculate_metrics(y_test, y_pred)
    
    for i in range(10):
        random_3_features = list(np.random.choice(X.columns, size=3, replace=False))
        X_random = X[random_3_features]
        X_train, X_test, y_train, y_test = train_test_split(
            X_random, y, 
            test_size=0.25, 
            random_state=42,
            stratify=stratify_col
        )
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        rf_model.fit(X_train_scaled, y_train)
        y_pred = rf_model.predict(X_test_scaled)
        results[f'Random 3 Features (trial {i+1})'] = calculate_metrics(y_test, y_pred)
    
    for i in range(10):
        random_1_feature = list(np.random.choice(X.columns, size=1, replace=False))
        X_random = X[random_1_feature]
        X_train, X_test, y_train, y_test = train_test_split(
            X_random, y, 
            test_size=0.25, 
            random_state=42,
            stratify=stratify_col
        )
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        rf_model.fit(X_train_scaled, y_train)
        y_pred = rf_model.predict(X_test_scaled)
        results[f'Random 1 Feature (trial {i+1})'] = calculate_metrics(y_test, y_pred)
    
    return results

In [21]:
phase4_results = ablation_study_phase4_random(X, y, stratify_col)
phase4_df = pd.DataFrame(phase4_results).T
phase4_df_sorted = phase4_df.sort_values('accuracy', ascending=False)
phase4_df_sorted

,accuracy,hallway_accuracy,kitchen_accuracy,lab_accuracy,meeting_room_accuracy
Random 5 Features (trial 1),1.000000,1.0,1.0,1.000000,1.0
Random 5 Features (trial 2),1.000000,1.0,1.0,1.000000,1.0
Random 5 Features (trial 3),1.000000,1.0,1.0,1.000000,1.0
Random 5 Features (trial 4),1.000000,1.0,1.0,1.000000,1.0
Random 5 Features (trial 5),1.000000,1.0,1.0,1.000000,1.0
Random 5 Features (trial 6),1.000000,1.0,1.0,1.000000,1.0
Random 5 Features (trial 7),1.000000,1.0,1.0,1.000000,1.0
Random 5 Features (trial 9),1.000000,1.0,1.0,1.000000,1.0
Random 5 Features (trial 10),1.000000,1.0,1.0,1.000000,1.0
Random 3 Features (trial 10),1.000000,1.0,1.0,1.000000,1.0


In [22]:
# I know what everyone's thinking, these results seem too good to be true, so I'm going to make sure they're real
print(X.columns.tolist())

print("'room' in features:", 'room' in X.columns)
print("'trial' in features:", 'trial' in X.columns)
print("'room_label' in features:", 'room_label' in X.columns)

['UO_Type_Mesh_allocated_sum', 'UO_Type_Mesh_allocated_count', 'UO_Type_Mesh_allocated_mean', 'UO_Type_Mesh_resident_sum', 'UO_Type_Mesh_resident_mean', 'UO_Type_GameObject_allocated_sum', 'UO_Type_GameObject_allocated_count', 'UO_Type_GameObject_allocated_mean', 'UO_Type_GameObject_resident_sum', 'UO_Type_GameObject_resident_mean', 'UO_Type_MeshFilter_allocated_sum', 'UO_Type_MeshFilter_allocated_count', 'UO_Type_MeshFilter_allocated_mean', 'UO_Type_MeshFilter_resident_sum', 'UO_Type_MeshFilter_resident_mean', 'UO_Type_MeshRenderer_allocated_sum', 'UO_Type_MeshRenderer_allocated_count', 'UO_Type_MeshRenderer_allocated_mean', 'UO_Type_MeshRenderer_resident_sum', 'UO_Type_MeshRenderer_resident_mean', 'UO_Type_MonoBehaviour_allocated_sum', 'UO_Type_MonoBehaviour_allocated_count', 'UO_Type_MonoBehaviour_allocated_mean', 'UO_Type_MonoBehaviour_resident_sum', 'UO_Type_MonoBehaviour_resident_mean', 'UO_Type_Transform_allocated_sum', 'UO_Type_Transform_allocated_count', 'UO_Type_Transform_all

In [24]:
# Check if any feature is perfectly correlated with room type
for col in X.columns:
    correlation = []
    for room_type in y.unique():
        room_mask = y == room_type
        correlation.append(X.loc[room_mask, col].values)
    
    if col in X.columns[:10]:
        print(f"\n{col}:")
        for i, room_type in enumerate(y.unique()):
            print(f"  {room_type}: min={correlation[i].min():.2f}, max={correlation[i].max():.2f}")


UO_Type_Mesh_allocated_sum:
  kitchen: min=0.52, max=0.58
  hallway: min=0.08, max=0.08
  lab: min=0.62, max=0.66
  meeting_room: min=0.29, max=0.30

UO_Type_Mesh_allocated_count:
  kitchen: min=21.00, max=23.00
  hallway: min=12.00, max=12.00
  lab: min=60.00, max=60.00
  meeting_room: min=39.00, max=39.00

UO_Type_Mesh_allocated_mean:
  kitchen: min=0.02, max=0.03
  hallway: min=0.01, max=0.01
  lab: min=0.01, max=0.01
  meeting_room: min=0.01, max=0.01

UO_Type_Mesh_resident_sum:
  kitchen: min=0.49, max=0.54
  hallway: min=0.07, max=0.07
  lab: min=0.56, max=0.56
  meeting_room: min=0.24, max=0.24

UO_Type_Mesh_resident_mean:
  kitchen: min=0.02, max=0.03
  hallway: min=0.01, max=0.01
  lab: min=0.01, max=0.01
  meeting_room: min=0.01, max=0.01

UO_Type_GameObject_allocated_sum:
  kitchen: min=0.00, max=0.00
  hallway: min=0.00, max=0.00
  lab: min=0.00, max=0.00
  meeting_room: min=0.00, max=0.00

UO_Type_GameObject_allocated_count:
  kitchen: min=250.00, max=254.00
  hallway: mi

In [25]:
# Check for features with completely non-overlapping ranges
perfectly_separable_features = []

for col in X.columns:
    ranges = []
    for room_type in y.unique():
        room_mask = y == room_type
        min_val = X.loc[room_mask, col].min()
        max_val = X.loc[room_mask, col].max()
        ranges.append((room_type, min_val, max_val))
    
    # Check if ranges are completely non-overlapping
    # For each pair of room types, check if their ranges don't overlap
    is_separable = True
    for i in range(len(ranges)):
        for j in range(i+1, len(ranges)):
            room1, min1, max1 = ranges[i]
            room2, min2, max2 = ranges[j]
            # Check if ranges overlap: max1 >= min2 and max2 >= min1
            if max1 >= min2 and max2 >= min1:
                is_separable = False
                break
        if not is_separable:
            break
    
    if is_separable:
        perfectly_separable_features.append(col)
        print(f"\n{col}:")
        for room_type, min_val, max_val in ranges:
            print(f"  {room_type}: min={min_val:.2f}, max={max_val:.2f}")

print(f"\n\nTotal perfectly separable features: {len(perfectly_separable_features)}")
print(f"Out of {len(X.columns)} total features")
print(f"Percentage: {100*len(perfectly_separable_features)/len(X.columns):.1f}%")


UO_Type_Mesh_allocated_sum:
  kitchen: min=0.52, max=0.58
  hallway: min=0.08, max=0.08
  lab: min=0.62, max=0.66
  meeting_room: min=0.29, max=0.30

UO_Type_Mesh_allocated_count:
  kitchen: min=21.00, max=23.00
  hallway: min=12.00, max=12.00
  lab: min=60.00, max=60.00
  meeting_room: min=39.00, max=39.00

UO_Type_Mesh_allocated_mean:
  kitchen: min=0.02, max=0.03
  hallway: min=0.01, max=0.01
  lab: min=0.01, max=0.01
  meeting_room: min=0.01, max=0.01

UO_Type_Mesh_resident_sum:
  kitchen: min=0.49, max=0.54
  hallway: min=0.07, max=0.07
  lab: min=0.56, max=0.56
  meeting_room: min=0.24, max=0.24

UO_Type_Mesh_resident_mean:
  kitchen: min=0.02, max=0.03
  hallway: min=0.01, max=0.01
  lab: min=0.01, max=0.01
  meeting_room: min=0.01, max=0.01

UO_Type_GameObject_allocated_count:
  kitchen: min=250.00, max=254.00
  hallway: min=232.00, max=232.00
  lab: min=328.00, max=328.00
  meeting_room: min=286.00, max=286.00

UO_Type_MeshFilter_allocated_count:
  kitchen: min=197.00, max=19

In [26]:
# After split, verify no overlap
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.25, 
    random_state=42, 
    stratify=stratify_col
)

print(f"Train indices: {X_train.index[:10].tolist()}")
print(f"Test indices: {X_test.index[:10].tolist()}")
print(f"Any overlap: {len(set(X_train.index) & set(X_test.index))}")

Train indices: [95, 58, 91, 86, 63, 8, 57, 1, 4, 74]
Test indices: [85, 82, 51, 24, 67, 52, 83, 31, 90, 72]
Any overlap: 0
